# 04 — Threshold tuning e analisi errori

## Obiettivi didattici

1. Tracciare **curva PR** e **curva ROC** del miglior modello.
2. Ottimizzare la **soglia decisionale** in base alla matrice dei costi.
3. Analizzare le **confusion matrix** alla soglia 0.5 vs ottimale.
4. Esaminare **feature importance**.
5. Esporre il modello tramite `predict_fraud()`.
!!! note "Dataset richiesto"
    Il dataset Kaggle (~470MB) NON e' in repo per limiti di GitHub.
    Scaricalo da <https://www.kaggle.com/datasets/kartik2112/fraud-detection>
    e copia `fraudTrain.csv` e `fraudTest.csv` in `data/raw/`.


In [ ]:
import sys; sys.path.insert(0, '../src')
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import joblib
from pathlib import Path

from fraud_pipeline.data import load_train_test, split_features_target
from fraud_pipeline.threshold import (
    CostMatrix, optimal_threshold_by_cost, threshold_sweep,
    confusion_matrix_at_threshold,
)
from fraud_pipeline.evaluation import (
    compute_metrics, plot_pr_curve, plot_roc_curve, plot_confusion_matrix,
    get_top_feature_importance, plot_feature_importance,
    make_classification_report,
)
from fraud_pipeline.inference import predict_fraud, example_transaction


## Caricamento test set + miglior modello

In [ ]:
_, df_test = load_train_test()
X_test, y_test = split_features_target(df_test)

models_dir = Path('../reports/models')
model_paths = list(models_dir.glob('*_best.joblib'))
models = {p.stem.replace('_best', ''): joblib.load(p) for p in model_paths}
list(models.keys())


## Metriche su holdout test (a soglia default 0.5)

Riepilogo per ogni modello: precision, recall, F1, F2, AUC-PR, AUC-ROC.

In [ ]:
rows = []
probas = {}
for name, model in models.items():
    proba = model.predict_proba(X_test)[:, 1]
    probas[name] = proba
    m = compute_metrics(y_test.values, proba, threshold=0.5)
    rows.append({'model': name, **m.as_dict()})
metrics_df = pd.DataFrame(rows).sort_values('auc_pr', ascending=False).set_index('model')
metrics_df.style.format({
    'precision': '{:.3f}', 'recall': '{:.3f}',
    'f1': '{:.3f}', 'f2': '{:.3f}',
    'auc_pr': '{:.4f}', 'auc_roc': '{:.4f}',
})


## Curve PR e ROC del miglior modello

**PR curve** mostra il vero trade-off su problemi sbilanciati. La linea tratteggiata rappresenta il random classifier (precision = prevalenza).

In [ ]:
best_name = metrics_df.index[0]
print(f'Best model: {best_name}')
best_proba = probas[best_name]
fig1 = plot_pr_curve(y_test.values, best_proba,
                     title=f'{best_name}: Precision-Recall (test)')
plt.show()
fig2 = plot_roc_curve(y_test.values, best_proba,
                      title=f'{best_name}: ROC (test)')
plt.show()


## Ottimizzazione soglia su matrice di costi

Cost matrix di esempio:
- **Costo False Negative (frode mancata)**: $120 (perdita media stimata).
- **Costo False Positive (legit bloccata)**: $5 (chargeback friction).

La funzione `optimal_threshold_by_cost` trova la soglia che minimizza il costo atteso totale.

In [ ]:
cost = CostMatrix(cost_fn=120.0, cost_fp=5.0)
best_threshold, best_cost = optimal_threshold_by_cost(
    y_test.values, best_proba, cost=cost,
)
print(f'Soglia ottimale: t = {best_threshold:.4f}')
print(f'Costo atteso  : ${best_cost:,.2f}')


## Sweep delle soglie

Visualizziamo precision/recall/F1/F2/cost al variare della soglia per capire dove si trova il punto di equilibrio.

In [ ]:
sweep = threshold_sweep(y_test.values, best_proba, cost=cost)
fig, ax1 = plt.subplots(figsize=(11, 5))
ax1.plot(sweep.threshold, sweep.precision, label='precision', color='#4C72B0')
ax1.plot(sweep.threshold, sweep.recall, label='recall', color='#C44E52')
ax1.plot(sweep.threshold, sweep.f1, label='F1', color='#55A868', linestyle='--')
ax1.plot(sweep.threshold, sweep.f2, label='F2', color='#8172B2', linestyle='--')
ax1.set_xlabel('threshold')
ax1.set_ylabel('metric value')
ax1.legend(loc='center left')
ax2 = ax1.twinx()
ax2.plot(sweep.threshold, sweep.expected_cost, color='black', alpha=0.6,
         label='cost')
ax2.axvline(best_threshold, color='red', linestyle=':', label=f't*={best_threshold:.3f}')
ax2.set_ylabel('expected_cost ($)')
ax2.legend(loc='center right')
plt.title('Threshold sweep'); plt.show()


## Confronto confusion matrix: 0.5 vs ottimale

Vediamo l'effetto pratico dello spostamento della soglia.

In [ ]:
for t, label in [(0.5, 'default'), (best_threshold, 'optimal')]:
    cm = confusion_matrix_at_threshold(y_test.values, best_proba, t)
    print(f'Soglia {label} = {t:.4f}: {cm}')
    y_pred = (best_proba >= t).astype(int)
    plot_confusion_matrix(y_test.values, y_pred,
                          title=f'{best_name} @t={t:.3f} ({label})')
    plt.show()


## Classification report a soglia ottimale

In [ ]:
y_pred_opt = (best_proba >= best_threshold).astype(int)
print(make_classification_report(y_test.values, y_pred_opt))


## Feature importance del miglior modello (se tree-based)

In [ ]:
model = models[best_name]
try:
    # Recupera nomi feature post-preprocessor.
    fe = model.named_steps['feature_engineer']
    preproc = model.named_steps['preprocessor']
    feature_names = preproc.get_feature_names_out()
    # Per estrarre feature_importances_, dobbiamo passargli la pipeline
    # del modello finale, non quella esterna.
    inner = type(model)(steps=[('preprocessor', preproc),
                                ('model', model.named_steps['model'])])
    df_imp = get_top_feature_importance(inner, feature_names, top_n=20)
    print(df_imp.to_string(index=False))
    plot_feature_importance(df_imp, title=f'{best_name}: top-20 feature')
    plt.show()
except Exception as e:
    print(f'Feature importance non estratta: {e}')


## API di inferenza: `predict_fraud()`

Il modello e' ora servito tramite una funzione semplice. Accetta dict (singola transazione) o DataFrame (batch). La soglia ottimale e' caricata automaticamente da `reports/models/threshold.json`.

In [ ]:
tx = example_transaction()
result = predict_fraud(tx, threshold=best_threshold)
print(result)

# Batch su 5 transazioni del test set:
batch_results = predict_fraud(X_test.head(5), threshold=best_threshold)
for i, r in enumerate(batch_results):
    actual = int(y_test.iloc[i])
    print(f'  tx[{i}]: prob={r["fraud_probability"]:.4f}  '
          f'pred={int(r["is_fraud"])}  actual={actual}')


## Conclusione

Il pipeline e' ora **production-ready**:

1. Modello + threshold serializzati in `reports/models/`.
2. `predict_fraud()` come unico punto di ingresso per l'inferenza.
3. Metriche e analisi errori in `reports/`.
4. Soglia decisionale ottimizzata sulla matrice di costi del business.

Per dettagli teorici (perche' AUC-PR, perche' time-series CV, perche' log-amt e z-score, ...) vedi `docs/teoria/`.
